In [1]:
# Load necessary libraries
import pandas as pd
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [2]:
# Load dataset
df = pd.read_csv('C:\\Project\\Machine Learning Engineer Project\\Shopee Review Sentiment Analysis\\data\\data.csv')

In [3]:
# Display the first few rows of the dataset
df.head()

,comment,rating_star
0,Absorption:belum cuba\nSkin Suitability:belum ...,0
1,Barang diterima dengan baik kualiti tidak ok s...,0
2,"Just received , but unfortunately the product ...",0
3,Skin Suitability:good\nAbsorption:good\n\nPeng...,0
4,Absorption:good\nSkin Suitability:good\n\ntak ...,0


In [4]:
# Display dataset information
df.describe()

,rating_star
count,9145.000000
mean,1.129470
std,0.885797
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,2.000000


In [5]:
# Display dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9145 entries, 0 to 9144
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   comment      9145 non-null   object
 1   rating_star  9145 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 143.0+ KB


In [6]:
# Define a dictionary for common Malay slang and their standard forms
malay_slang_dict = {
    "tk": "tidak", "x": "tidak", "tak": "tidak", "xde": "tak ada",
    "tp": "tapi", "sgt": "sangat", "mcm": "macam", "jgk": "juga", "jg": "juga",
    "brg": "barang", "nk": "nak", "dh": "dah", "smpai": "sampai", 
    "yg": "yang", "bkn": "bukan", "blm": "belum", "utk": "untuk",
    "penghntran": "penghantaran", "sdp": "sedap", "giler": "gila"
}

In [7]:
# Function to clean text
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower().replace('\n', ' ')
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [malay_slang_dict.get(t, t) for t in tokens]

    return " ".join(tokens)

In [8]:
# Apply cleaning function
df['cleaned_comment'] = df['comment'].apply(clean_text)

In [ ]:
# Defines the input features and the target labels
X = df['cleaned_comment']
y = df['rating_star'] 

In [10]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
# Vectorization and model training
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

In [12]:
# Transform data
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [13]:
# Train logistic regression model
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [14]:
# Evaluate model
y_pred = model.predict(X_test_tfidf)

In [15]:
# Print evaluation metrics
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred))

Accuracy: 0.72
              precision    recall  f1-score   support

           0       0.75      0.75      0.75       614
           1       0.44      0.54      0.48       364
           2       0.86      0.77      0.81       851

    accuracy                           0.72      1829
   macro avg       0.68      0.69      0.68      1829
weighted avg       0.74      0.72      0.73      1829



In [16]:
# Function to predict sentiment of new comments
def predict_new_comment(text):
    cleaned = clean_text(text)
    vector = tfidf.transform([cleaned])
    prediction = model.predict(vector)[0]
    
    label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
    return label_map[prediction]

In [17]:
# Test the prediction function
test_samples = [
    "Barang sampai lambat gila, xde bubble wrap, box hancur!", 
    "Item received in good condition, sangat berpuas hati.",  
    "Boleh lah, rasa okay je tapi sikit sangat."               
]


In [18]:
# Predict sentiments for test samples
for s in test_samples:
    print(f"Original comment: {s}")
    print(f"Predicted sentiment: {predict_new_comment(s)}\n")

Original comment: Barang sampai lambat gila, xde bubble wrap, box hancur!
Predicted sentiment: Negative

Original comment: Item received in good condition, sangat berpuas hati.
Predicted sentiment: Positive

Original comment: Boleh lah, rasa okay je tapi sikit sangat.
Predicted sentiment: Neutral



In [19]:
# Save the model and vectorizer
joblib.dump(model, 'sentiment_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']